# Лабораторная работа 2. Борьба со спамом
## Часть 2. Мешок слов и наивный байесовский классификатор

**Выполнил(а):** Михаил Осколков, Антон Бердников группа ИНБс-5301

**Что делаем в этой части:**
1. Переводим тексты писем в числовые векторы — представление **«мешок слов»** (`CountVectorizer`).
2. Обучаем **наивный байесовский классификатор** (`MultinomialNB`) из библиотеки scikit-learn.
3. Сравниваем его качество с чёрным списком из части 1 и делаем выводы.

> Перед началом должна быть выполнена часть 1 (`01. Разбор писем и чёрный список.ipynb`): понадобятся тот же датасет и та же функция чтения писем.

## Шаг 0. Подготовка (как в части 1)

Папка `datasets` должна лежать рядом с блокнотом. В Google Colab загрузите датасет так же, как в части 1 — раскомментируйте нужный вариант в ячейке ниже (локально ничего делать не нужно).

In [1]:
# --- Google Colab, вариант А: архив лежит в корне Google Диска ---
from google.colab import drive
drive.mount('/content/drive')
!apt-get install -y p7zip-full > /dev/null
!7z x -p1 -o/content/ "/content/drive/MyDrive/datasets.zip" -y

# --- Google Colab, вариант Б: загрузка архива в текущую сессию ---
# from google.colab import files
# uploaded = files.upload()          # выберите datasets.zip
# !apt-get install -y p7zip-full > /dev/null
# !7z x -p1 datasets.zip -y

Mounted at /content/drive

7-Zip 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=en_US.UTF-8 Threads:2 OPEN_MAX:1048576

Scanning the drive for archives:
  0M Scan /content/drive/MyDrive/                                 1 file, 39644509 bytes (38 MiB)

Extracting archive: /content/drive/MyDrive/datasets.zip
 40% 4096 Open              --
Path = /content/drive/MyDrive/datasets.zip
Type = zip
Physical Size = 39644509

  0%      9% 883         18% 1868          28% 2734 - datasets/trec07delay/data/inmail.3456                                                  38% 3655          47% 4707          58% 5785          69% 6916          77% 7802         

In [2]:
import os
import email
from email import policy
import pandas as pd

DATA_DIR = 'datasets/trec07delay/data/'
LABELS_FILE = 'datasets/trec07delay/delay/index'

assert os.path.isdir(DATA_DIR) and os.path.isfile(LABELS_FILE), \
    'Датасет не найден — распакуйте datasets.zip рядом с блокнотом (шаг 0)'

def text_email(path):
    """Извлекает тему и текстовое (text/plain) тело письма — та же функция, что в части 1."""
    try:
        with open(path, 'rb') as f:
            msg = email.message_from_binary_file(f, policy=policy.default)
        subject = str(msg.get('Subject') or '')
        parts = []
        for part in msg.walk():
            if part.get_content_type() == 'text/plain':
                try:
                    content = part.get_content()
                    if isinstance(content, str):
                        parts.append(content)
                except Exception:
                    pass
        return subject + ' ' + ' '.join(parts)
    except Exception:
        return ''

# Читаем метки: имя файла → 'spam'/'ham'
labels = {}
with open(LABELS_FILE) as f:
    for line in f:
        parts = line.split()
        if len(parts) == 2:
            labels[parts[1].split('/')[-1]] = parts[0]

print('Меток прочитано:', len(labels))

Меток прочитано: 10000


In [3]:
# Читаем все письма в список текстов X и список меток y (займёт до минуты)
X, y = [], []
empty = 0
filelist = sorted(labels, key=lambda name: int(name.split('.')[1]))
for filename in filelist:
    text = text_email(DATA_DIR + filename)
    if len(text.split()) <= 2:      # отсев пустых и HTML-only писем по сырому тексту;
        empty += 1                  # в части 1 он строже (после удаления стоп-слов), поэтому числа отличаются
        continue
    X.append(text)
    y.append(labels[filename])

print(f'Прочитано писем: {len(X)}, пропущено пустых: {empty}')
print('Пример текста (первые 200 символов):')
print(X[0][:200])

Прочитано писем: 9868, пропущено пустых: 132
Пример текста (первые 200 символов):
Generic Cialis, branded quality@  


## Шаг 1. Мешок слов: превращаем текст в числа

Модели машинного обучения работают с числами, а не с текстом. Самое простое числовое представление текста — **«мешок слов» (bag-of-words)**:

1. Проходим по всем документам и собираем **словарь** — список всех встретившихся слов.
2. Каждый документ представляем вектором длиной со словарь: на i-м месте стоит **число вхождений** i-го слова в этот документ.

Порядок слов и грамматика при этом теряются — остаётся только «какие слова и сколько раз встретились». Для классификации спама этого, как ни странно, почти достаточно. Разберём на простом примере из трёх «писем»:

In [6]:
from sklearn.feature_extraction.text import CountVectorizer

toy = ['hello mr bear',
       'hello hello gunter',
       'goodbye mr gunter']

toy_vectorizer = CountVectorizer()
toy_counts = toy_vectorizer.fit_transform(toy)

print('Словарь:', list(toy_vectorizer.get_feature_names_out()))
print(toy_counts.toarray())   # строки — письма, столбцы — слова словаря

Словарь: ['bear', 'goodbye', 'gunter', 'hello', 'mr']
[[1 0 0 1 1]
 [0 0 1 2 0]
 [0 1 1 0 1]]


Столбцы идут в алфавитном порядке словаря (`bear`, `goodbye`, `gunter`, `hello`, `mr`), поэтому первая строка `[1 0 0 1 1]` читается так: в письме A по одному разу встречаются `bear`, `hello` и `mr`, а `goodbye` и `gunter` — ни разу.

В реальном корпусе словарь содержит сотни тысяч слов, а в каждом отдельном письме встречается лишь малая их часть — матрица почти целиком нулевая. Поэтому `CountVectorizer` хранит её в **разреженном (sparse)** формате: в памяти лежат только ненулевые элементы.

In [7]:
# Случайное разбиение 70/30; stratify=y сохраняет долю спама одинаковой в обеих частях
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.7, random_state=2, stratify=y)

print(f'train: {len(X_train)} писем, test: {len(X_test)} писем')
print('Доля спама: train {:.1%}, test {:.1%}'.format(
    sum(v == 'spam' for v in y_train) / len(y_train),
    sum(v == 'spam' for v in y_test) / len(y_test)))

train: 6907 писем, test: 2961 писем
Доля спама: train 80.3%, test 80.3%


In [8]:
# Строим словарь ПО ТРЕНИРОВОЧНОЙ ВЫБОРКЕ и векторизуем обе части
vectorizer = CountVectorizer()
X_train_counts = vectorizer.fit_transform(X_train)   # fit: составить словарь + посчитать слова
X_test_counts = vectorizer.transform(X_test)         # только посчитать — словарь уже зафиксирован

print('Размер словаря:', len(vectorizer.vocabulary_))
print('Матрица train:', X_train_counts.shape, '| ненулевых элементов:', X_train_counts.nnz)
print('Матрица test: ', X_test_counts.shape)

Размер словаря: 45547
Матрица train: (6907, 45547) | ненулевых элементов: 462165
Матрица test:  (2961, 45547)


> **Важно:** `fit_transform` вызывается только на train — словарь строится по тренировочным данным. Тестовая выборка проходит только через `transform`: слова, которых не было в train, просто игнорируются. Иначе информация о тесте «протечёт» в обучение.

## Шаг 2. Наивный байесовский классификатор

Идея: для каждого класса (спам / ham) по тренировочной выборке оценивается, с какой вероятностью в нём встречается каждое слово словаря. Для нового письма по **формуле Байеса** вычисляется, к какому классу оно вероятнее относится, — по совокупности его слов.

Классификатор называется «наивным», потому что слова считаются независимыми друг от друга (в реальном языке это, конечно, не так). На практике для текстов метод работает на удивление хорошо, а обучается за доли секунды.

`MultinomialNB` — вариант наивного Байеса для счётчиков слов (наш случай). Чтобы слово, не встречавшееся в train, не обнулило всё произведение вероятностей, применяется **сглаживание Лапласа** (параметр `alpha`, по умолчанию 1.0). Подробная теория — в Приложении А методички.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train_counts, y_train)     # обучение: подсчёт вероятностей слов в каждом классе
y_pred = model.predict(X_test_counts)  # прогноз для тестовой выборки

print('Первые 10 прогнозов:      ', [str(v) for v in y_pred[:10]])
print('Первые 10 истинных меток: ', y_test[:10])

Первые 10 прогнозов:       ['spam', 'ham', 'spam', 'spam', 'spam', 'ham', 'ham', 'ham', 'spam', 'spam']
Первые 10 истинных меток:  ['spam', 'ham', 'spam', 'spam', 'spam', 'ham', 'ham', 'ham', 'spam', 'spam']


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print(f'Accuracy: {accuracy_score(y_test, y_pred):.1%}\n')
print(classification_report(y_test, y_pred, digits=3))

cm = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=['spam', 'ham']),
    index=['на самом деле спам', 'на самом деле ham'],
    columns=['прогноз: спам', 'прогноз: ham'])
cm

Accuracy: 97.9%

              precision    recall  f1-score   support

         ham      0.924     0.974     0.948       584
        spam      0.994     0.980     0.987      2377

    accuracy                          0.979      2961
   macro avg      0.959     0.977     0.968      2961
weighted avg      0.980     0.979     0.979      2961



,прогноз: спам,прогноз: ham
на самом деле спам,2330,47
на самом деле ham,15,569


Напоминание по столбцам отчёта `classification_report`:

- **precision** — какая доля писем, названных классом, действительно принадлежит ему;
- **recall** — какая доля писем класса была найдена;
- **f1-score** — гармоническое среднее precision и recall;
- **support** — сколько писем этого класса было в тесте.

## Шаг 3. Сравниваем два подхода

Подставьте в таблицу значения из своих запусков (часть 1 и часть 2):

| Подход | Accuracy | Precision (спам) | Recall (спам) | Чем примечателен |
|---|---|---|---|---|
| Чёрный список (часть 1) | ~50.3 % | ~85.9 % | ~49.1 % | Прозрачные правила, но «мёртвый» список слов |
| Наивный Байес (часть 2) | ~97.9 % | ~99.4 % | ~98 % | Обучается на данных сам, использует все слова |

**Выводы** *(заполните сами, 3–5 предложений)*: чем наивный Байес принципиально отличается от чёрного списка? Почему он заметно точнее? В чём его слабые места — что произойдёт, если спамеры изменят лексику?

Чёрный список работает тривиально: если в письме встретилось слово из списка, то это спам. Но спамеры пишут разными словами, поэтому так удаётся поймать только половину спама.
Наивный Байес работает умнее. Он сначала учится на примерах: смотрит, какие слова чаще бывают в спаме, а какие в обычных письмах. Потом для нового письма он оценивает все слова вместе и решает, больше оно похоже на спам или на легитимное письмо. Поэтому у него точность 97.9%, а у чёрного списка 50.3%.
Главное отличие: чёрный список это жёсткое правило и готовый набор слов. А Байес это модель, которая сама находит закономерности в данных. Поэтому он лучше обобщает и ловит спам, который не попал в чёрный список.
Минусы у Байеса тоже есть. Он наивно считает, что все слова в письме независимы друг от друга, хотя на самом деле они связаны. Ещё он сильно зависит от качества данных: если спам сильно изменится, модель придётся переобучать.

## Что дальше

Полученный результат — уровень простого, но реального спам-фильтра, и достигнут он несколькими строками кода. Дальнейшие улучшения разобраны в индивидуальных вариантах (Приложение Г методички): TF-IDF вместо счётчиков, другие классификаторы, биграммы, анализ ошибок и т. д.

**Не забудьте сдать работу:** скачайте оба блокнота (Файл → Скачать → Скачать .ipynb), положите их в папку `lab2/` локального репозитория, добавьте блокнот с индивидуальным вариантом, сделайте коммит и отправьте на gitflic.ru (методичка, шаг 10).